# SPAI phase-two training with CvT-13 — NIH chest X-rays vs. synthetic chest X-rays

This notebook fine-tunes the phase-two SPAI detector heads on top of the **frozen phase-one
CvT-13 MFM encoder**, comparing a **fixed** frequency-masking radius against a **learnable**
one (see [docs/learnable_radius.md](docs/learnable_radius.md)), using two Kaggle datasets:

| Role | Dataset | Images |
|---|---|---|
| Authentic → class `0` | [`nih-chest-xrays/data`](https://www.kaggle.com/datasets/nih-chest-xrays/data) | ~112,120 |
| Generated → class `1` | [`whiteflags26/synthetic-chest-x-rays`](https://www.kaggle.com/datasets/whiteflags26/synthetic-chest-x-rays) | ~8,550 |

Repository: https://github.com/Kashshaf-Labib/spai-parameterized-radius-cvt

## Before you run — Kaggle setup

1. **Add Input** → search `nih-chest-xrays/data` → **Add**.
2. **Add Input** → search `whiteflags26/synthetic-chest-x-rays` → **Add**.
3. **Settings → Accelerator** → `GPU T4 x2` (only GPU 0 is used; T4 x2 is Kaggle's standard
   free GPU offering).
4. **Settings → Internet** → **On** (required to clone the repo, `pip install`, and download
   the phase-one checkpoint from Google Drive).
5. Run top to bottom. **Sections 2, 4, 6 and 7 are the gate** — dataset discovery, the
   acquisition-parity audit, the architecture smoke test, and the pilot training run. If any of
   them fails, stop and fix it before spending GPU hours on the full run in section 8.

## Running this unattended (so it survives your laptop sleeping / going offline)

An **interactive "Edit" session is the wrong tool for a multi-hour run** — it is tied to your
browser tab, and Kaggle can reclaim a session that has gone idle/disconnected for a while, which
would kill training. For anything beyond the pilot, use:

**Save Version → Save & Run All (Commit)** (top-right of the notebook editor), with "Always save
output" checked. This queues the *entire* notebook as a background batch job on Kaggle's own
servers — it keeps running whether your laptop is on, asleep, or offline. Kaggle notifies you
when it finishes or fails; you don't need to watch it.

Two consequences worth knowing before launching the full run in section 8:

- **GPU sessions are time-boxed** (Kaggle currently allows several hours per session plus a
  weekly quota that changes over time — check the current numbers under kaggle.com/settings).
  Section 8's duration projection estimates whether your schedule fits *before* you commit to it.
- **Checkpoints auto-resume within one session's `/kaggle/working` automatically** (see
  "Checkpointing & resuming" in section 8) — re-running the same Version after an interruption
  just continues. Continuing in a **brand-new** session (e.g. the previous Commit already
  finished or expired) needs one extra step, also covered there: attach its output as a new
  Input dataset and point `PREVIOUS_FIXED_CHECKPOINT` / `PREVIOUS_LEARNABLE_CHECKPOINT` at it.

## 0 · Parameters

Everything configurable lives here. `REAL_DIR` / `FAKE_DIR` are best guesses for where Kaggle
mounts the two datasets — section 2 lists what is actually mounted, so correct these first if
it disagrees.

In [ ]:
from pathlib import Path

# --------------------------------------------------------------------- source code & weights
REPO_URL    = "https://github.com/Kashshaf-Labib/spai-parameterized-radius-cvt.git"
REPO_BRANCH = "main"
WEIGHT_URL      = "https://drive.google.com/file/d/180hbXuIVAkXE3iT8BQO3ivoV1mBjz-r7/view?usp=drive_link"
EXPECTED_SHA256 = "016a6ae03fcf4297803907d354ad519ecc406867586e2e6e7fac391f32264c33"

# ----------------------------------------------------------------------------- input data
# Kaggle's mount point does not always match the dataset slug 1:1 - on this account, "Add
# Input" nests both datasets under /kaggle/input/datasets/<owner>/<slug> instead of the more
# common /kaggle/input/<slug>. Section 2 lists every directory under /kaggle/input that
# actually holds images - correct these against what it prints if they are wrong.
REAL_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"                    # -> class 0
FAKE_DIR = "/kaggle/input/datasets/whiteflags26/synthetic-chest-x-rays"     # -> class 1

# ------------------------------------------------------------------ acquisition parity
# Real and generated images must differ ONLY in content. If they also differ in resolution,
# file format or colour mode, the detector can separate them on that shortcut alone and the
# measured AUC says nothing about generated-image detection. Section 4 audits this; these
# settings (applied by section 5/8 while building the datasets) remove whatever it finds.
TARGET_SIZE = 512      # bring both classes to this size
RESIZE_MODE = "crop"   # "crop": center-crop images already large enough, leaving their
                        #         spectrum untouched (resampling is a low-pass filter that
                        #         attenuates exactly the high frequencies SPAI depends on).
                        # "resize": resample everything to a comparable scale instead, at
                        #         the cost of altering the spectrum.
RECODE  = "png"        # re-encode both classes into one file format ("png" or "jpeg")
TO_GRAY = True          # discard chrominance - X-rays carry none

# ------------------------------------------------------------------------------- splits
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.80, 0.10, 0.10
# NIH filenames are <patient_id>_<followup>.png. Grouping by patient keeps every image of a
# patient inside a single split, so the same patient can't leak from train into test.
# spai.tools.prepare_medical_dataset falls back to one group per image when this doesn't
# apply cleanly to a class (e.g. the synthetic images, which have no patient concept).
GROUP_REGEX = r"^(?P<group>[^_]+)_"
SEED = 0

# ---------------------------------------------------------------------------- pilot run
# Kept tiny and fast on purpose: this is the "does the whole pipeline run" gate, not a
# measurement of model quality.
PILOT_MAX_PER_CLASS = 150      # per class -> ~240 train / ~30 val / ~30 test images
PILOT_EPOCHS = 2
PILOT_WARMUP = 1
PILOT_BATCH  = 4
PILOT_ACCUM  = 2
PILOT_LR     = 2.2e-4

# ---------------------------------------------------------------------------- full run
# Both classes are capped to whichever is smaller, so the effective ceiling is the number of
# generated images (~8550). Set to None to use every image (mind the disk estimate printed
# in section 8, and the ~20 GB /kaggle/working cap).
FULL_MAX_PER_CLASS = 8000
FULL_EPOCHS = 35    # the paper's schedule
FULL_WARMUP = 5
# CvT-13 is far smaller than the paper's ViT-B backbone, but the learnable radius keeps the
# frozen backbone's autograd graph alive (see docs/learnable_radius.md) and needs more memory
# per sample, so it uses a smaller per-step batch at the same effective batch size.
FULL_BATCH_FIXED,     FULL_ACCUM_FIXED     = 8, 4    # effective batch 32
FULL_BATCH_LEARNABLE, FULL_ACCUM_LEARNABLE = 4, 8    # effective batch 32
# The paper uses lr 5e-4 at batch 72. Scaled linearly to an effective batch of 32.
FULL_LR  = 2.2e-4
FULL_AMP = "O0"     # APEX is not installed on Kaggle - O0 (fp32) is the only supported value.

# Explicit checkpoints to resume from in a BRAND NEW Kaggle session (e.g. a previous Commit's
# output, re-attached as an Input dataset). Leave as None for a first run, or when continuing
# training within the SAME session/working directory - that case is handled automatically by
# the training CLI's own auto-resume (see section 8's "Checkpointing & resuming" note).
PREVIOUS_FIXED_CHECKPOINT     = None   # e.g. "/kaggle/input/.../ckpt_epoch_12.pth"
PREVIOUS_LEARNABLE_CHECKPOINT = None

# ------------------------------------------------------------------------------- runtime
DATA_WORKERS             = 2
FEATURE_EXTRACTION_BATCH = 128   # bounds VRAM on the any-resolution val/test path
VAL_BATCH                = 8

# -------------------------------------------------------------------------------- layout
WORK     = Path("/kaggle/working")
REPO_DIR = WORK / "spai-parameterized-radius-cvt"
WEIGHT_PATH = REPO_DIR / "weights" / "cvt_mfm_pretrain.pth"

PILOT_DS = WORK / "datasets" / "pilot"
FULL_DS  = WORK / "datasets" / "full"
PILOT_OUT = WORK / "output" / "pilot"
FULL_OUT  = WORK / "output" / "full"

# Stable (non-timestamped) tags. This matters: the training CLI auto-resumes from the newest
# checkpoint already present under <output>/<model_name>/<tag> whenever --resume is not
# passed - a fresh timestamp on every run would defeat that and always restart from epoch 0.
PILOT_TAG = "pilot"
FULL_TAG  = "full"

RADIUS_CONFIGS = [
    {
        "key": "fixed",
        "label": "Fixed radius",
        "cfg": "configs/spai_cvt.yaml",
        "model_name": "finetune_cvt",
        "pilot_batch": PILOT_BATCH, "pilot_accum": PILOT_ACCUM,
        "full_batch": FULL_BATCH_FIXED, "full_accum": FULL_ACCUM_FIXED,
        "previous_checkpoint": PREVIOUS_FIXED_CHECKPOINT,
    },
    {
        "key": "learnable",
        "label": "Learnable radius",
        "cfg": "configs/spai_cvt_learnable_radius.yaml",
        "model_name": "finetune_cvt_learnable_radius",
        "pilot_batch": PILOT_BATCH, "pilot_accum": PILOT_ACCUM,
        "full_batch": FULL_BATCH_LEARNABLE, "full_accum": FULL_ACCUM_LEARNABLE,
        "previous_checkpoint": PREVIOUS_LEARNABLE_CHECKPOINT,
    },
]

print("Parameters set. Repository ->", REPO_DIR)

## 1 · Environment

### 1.1 Clone the repository and install dependencies

In [ ]:
import os
import subprocess
import sys
import shutil

os.environ["DISABLE_NEPTUNE"] = "1"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


def run_command(*args):
    command = [str(a) for a in args]
    print("RUN:", subprocess.list2cmdline(command))
    subprocess.run(command, check=True)


def run_module(module, *args):
    run_command(sys.executable, "-m", module, *args)


if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    # A new Kaggle session can restore /kaggle/working from a prior Version's saved output,
    # which brings back the repo's files but not its .git metadata (dotfiles are typically
    # excluded) - leaving a directory that looks like the repo but isn't a checkout `git pull`
    # can use. Since everything under it is reproducible (the checkpoint re-downloads
    # automatically in section 3), it's safe to remove and re-clone.
    print(f"{REPO_DIR} exists but has no .git metadata - removing it and re-cloning.")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_command("git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR)
else:
    run_command("git", "-C", REPO_DIR, "pull", "--ff-only")
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
run_module("pip", "install", "-q", "-r", "requirements-kaggle.txt")

print("\nRepository at", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

### 1.2 Environment check

In [ ]:
import platform
import torch

print(f"python         : {platform.python_version()}")
print(f"torch          : {torch.__version__}  (CUDA {torch.version.cuda})")
print(f"GPU available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}        : {p.name}  {p.total_memory / 1024 ** 3:.1f} GiB")
else:
    raise RuntimeError("No GPU detected. Set Settings -> Accelerator -> GPU before continuing.")

import transformers
print(f"transformers   : {transformers.__version__}")
assert transformers.__version__ == "5.13.1", "The CvT integration depends on this pinned version."

for mod in ("timm", "albumentations", "torchmetrics"):
    try:
        print(f"{mod:<14} : {__import__(mod).__version__}")
    except Exception as e:
        print(f"{mod:<14} : MISSING ({e})")

total, _, free = shutil.disk_usage(WORK)
print(f"\n/kaggle/working free: {free / 1024 ** 3:.1f} GiB of {total / 1024 ** 3:.1f} GiB")

### 1.3 Helper functions

Used throughout the rest of the notebook.

In [ ]:
import re

import pandas as pd
import matplotlib.pyplot as plt


def run_dir(output_root, model_name, tag):
    '''Mirrors the training CLI's own <output>/<model_name>/<tag> layout.'''
    return Path(output_root) / model_name / tag


def full_run_dir(radius_config):
    '''Where a config's full-run checkpoints live, computed from section 0 alone.

    Deliberately independent of whether section 8's training cell for this config ran (to
    completion or at all) in the current kernel: it only reads FULL_OUT/model_name/FULL_TAG,
    all fixed at parameter time, and section 9/10 check the filesystem for checkpoints rather
    than an in-memory flag. That way stopping training early, evaluating in a separate
    session, or only ever training one of the two configs all work the same way.
    '''
    return run_dir(FULL_OUT / radius_config["key"], radius_config["model_name"], FULL_TAG)


def latest_checkpoint(directory):
    '''Returns the highest-epoch checkpoint in `directory`.

    Checkpoints are only written when validation loss improves (unless --save-all was
    used), so the highest-numbered one is always the best one seen so far.
    '''
    checkpoints = sorted(
        Path(directory).glob("ckpt_epoch_*.pth"),
        key=lambda p: int(re.search(r"ckpt_epoch_(\d+)", p.name).group(1)),
    )
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint found under {directory}")
    return checkpoints[-1]


def run_for_each_config(configs, fn):
    '''Runs fn(config) for every radius config, continuing past failures so both

    configurations get a chance to report their own problem in a single pass, rather than
    the second one being skipped because the first one raised.
    '''
    failures = []
    for config in configs:
        print(f"\n{'=' * 20} {config['label']} {'=' * 20}")
        try:
            fn(config)
        except Exception as e:
            print(f"[FAILED] {config['label']}: {e}")
            import traceback
            traceback.print_exc()
            failures.append(config["label"])
    if failures:
        raise RuntimeError(f"Failed for: {', '.join(failures)}. See tracebacks above.")


def train_cmd(cfg, data_csv, csv_root, output, tag, epochs, warmup, batch, accum, lr, amp,
              resume=None, save_all=False):
    cmd = [
        "train",
        "--cfg", cfg,
        "--batch-size", batch,
        "--accumulation-steps", accum,
        "--learning-rate", lr,
        "--pretrained", WEIGHT_PATH,
        "--output", output,
        "--data-path", data_csv,
        "--csv-root-dir", csv_root,
        "--tag", tag,
        "--amp-opt-level", amp,
        "--data-workers", DATA_WORKERS,
        "--opt", "TRAIN.EPOCHS", str(epochs),
        "--opt", "TRAIN.WARMUP_EPOCHS", str(warmup),
        "--opt", "DATA.VAL_BATCH_SIZE", str(VAL_BATCH),
        "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH),
        "--opt", "DATA.TEST_PREFETCH_FACTOR", "1",
        "--opt", "PRINT_FREQ", "5",
    ]
    if save_all:
        cmd.append("--save-all")
    if resume is not None:
        cmd.extend(("--resume", resume))
    return cmd


def parse_training_log(output_dir):
    '''Extracts per-epoch validation metrics from the training CLI's own text log.

    There is no separate history.csv in this codebase - log_rank0.txt (opened in append
    mode) already accumulates every epoch across resumed runs, so it doubles as one.
    '''
    log_path = Path(output_dir) / "log_rank0.txt"
    if not log_path.exists():
        raise FileNotFoundError(f"No training log at {log_path}")
    text = log_path.read_text(errors="ignore")

    patterns = {
        "val_loss": r"Val \| Epoch (\d+) \| Images: \d+ \| loss: ([\d.eE+-]+)",
        "val_accuracy": r"Val \| Epoch (\d+) \| Images: \d+ \| ACC: ([\d.eE+-]+)",
        "val_ap": r"Val \| Epoch (\d+) \| Images: \d+ \| AP: ([\d.eE+-]+)",
        "val_auc": r"Val \| Epoch (\d+) \| Images: \d+ \| AUC: ([\d.eE+-]+)",
        "masking_radius": r"Masking radius \| Epoch (\d+) \| r = ([\d.eE+-]+)",
    }
    records = {}
    for column, pattern in patterns.items():
        for epoch, value in re.findall(pattern, text):
            records.setdefault(int(epoch), {})[column] = float(value)
    for epoch, h, m, s in re.findall(r"EPOCH (\d+) training takes (\d+):(\d\d):(\d\d)", text):
        records.setdefault(int(epoch), {})["epoch_time"] = int(h) * 3600 + int(m) * 60 + int(s)
    if not records:
        raise RuntimeError(f"No epoch metrics found in {log_path}")
    return pd.DataFrame([{"epoch": e, **v} for e, v in sorted(records.items())])


def plot_history(history_or_dir, title):
    history = (parse_training_log(history_or_dir)
               if not isinstance(history_or_dir, pd.DataFrame) else history_or_dir)
    fig, (ax_loss, ax_met) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)

    ax_loss.plot(history.epoch, history.val_loss, marker="o", color="#2a78d6")
    best = int(history.val_loss.idxmin())
    ax_loss.scatter([history.epoch[best]], [history.val_loss[best]], s=50,
                     color="#eb6834", zorder=5)
    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("validation loss")
    ax_loss.set_title(f"Loss (best: epoch {int(history.epoch[best])})", loc="left")

    for column, label, color in (("val_auc", "AUC", "#2a78d6"), ("val_ap", "AP", "#eb6834"),
                                  ("val_accuracy", "accuracy", "#1baf7a")):
        if column in history:
            ax_met.plot(history.epoch, history[column], marker="o", color=color, label=label)
    ax_met.set_xlabel("epoch"); ax_met.set_ylabel("score"); ax_met.set_ylim(0, 1.02)
    ax_met.set_title("Validation metrics", loc="left")
    ax_met.legend()

    fig.suptitle(title, x=0.01, ha="left")
    plt.show()
    print(f"best val loss {history.val_loss.min():.4f} at epoch {int(history.epoch[best])} | "
          f"best val AUC {history.val_auc.max():.4f}")
    return history


def verify_parity(out_dir):
    '''Re-audits the arranged images, confirming the two classes now share their properties.

    Section 4 audits the source images; this describes what the model will actually train on.
    '''
    from spai.tools.prepare_medical_dataset import audit_images, find_images, report_parity

    print("\n=== Acquisition properties AFTER preparation (train split) ===")
    audits = {}
    for cls, label in (("0_real", "authentic (class 0)"), ("1_fake", "generated (class 1)")):
        audits[cls] = audit_images(find_images(Path(out_dir) / "train" / cls), 400)
        a = audits[cls]
        print(f"  {label}")
        for field in ("sizes", "formats", "modes"):
            rendered = ", ".join(f"{k}: {v}" for k, v in a[field].most_common(4))
            print(f"    {field:<9}: {rendered}")

    warnings = report_parity(audits["0_real"], audits["1_fake"])
    if warnings:
        for w in warnings:
            print(f"\n  [WARNING] {w}")
        print("\n  >>> The classes still differ - revisit TARGET_SIZE/RECODE/TO_GRAY in "
              "section 0 before training, or the result will not be interpretable.")
    else:
        print("\n  OK - the two classes now share their resolution, format and colour mode.")


def build_dataset(out_dir, max_per_class, tag):
    '''Arranges both classes into the 0_real/1_fake tree and writes the dataset CSVs.'''
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)

    prepare_args = [
        "--real-dir", REAL_DIR, "--fake-dir", FAKE_DIR, "-o", out_dir,
        "--train-ratio", TRAIN_RATIO, "--val-ratio", VAL_RATIO, "--test-ratio", TEST_RATIO,
        "--group-regex", GROUP_REGEX, "--seed", SEED,
    ]
    if max_per_class is not None:
        prepare_args += ["--max-per-class", max_per_class]
    if TARGET_SIZE is not None:
        prepare_args += ["--target-size", TARGET_SIZE, "--resize-mode", RESIZE_MODE]
    if RECODE is not None:
        prepare_args += ["--recode", RECODE]
    if TO_GRAY:
        prepare_args += ["--to-gray"]
    run_module("spai.tools.prepare_medical_dataset", *prepare_args)

    run_module("spai.tools.create_dir_csv",
               "--train_dir", out_dir / "train", "--val_dir", out_dir / "val",
               "-o", out_dir / "train_val.csv", "-r", out_dir)
    run_module("spai.tools.create_dir_csv",
               "--test_dir", out_dir / "test",
               "-o", out_dir / "test.csv", "-r", out_dir)

    counts = {}
    for name in ("train_val", "test"):
        df = pd.read_csv(out_dir / f"{name}.csv")
        print(f"\n{tag}/{name}.csv")
        print(df.groupby(["split", "class"]).size().rename("images").to_frame())
        counts[name] = df

    verify_parity(out_dir)
    return counts


print("Helpers ready.")

## 2 · Locate the attached datasets

Confirm the two paths printed below match `REAL_DIR` / `FAKE_DIR` in section 0. If they
differ, edit section 0 and re-run it, then re-run this cell.

In [ ]:
import collections
import pathlib

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def count_images_per_dir(root):
    '''Maps every directory under `root` to the number of images directly inside it.'''
    counts = collections.Counter()
    for p in pathlib.Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:
            counts[p.parent] += 1
    return counts


def describe_mounts(base="/kaggle/input", max_rows=25, max_depth=4):
    '''Reports every directory under the Kaggle mounts that actually contains images.

    Dataset mount points differ from their slugs often enough that hard-coding them is
    unreliable, so the candidates are discovered instead of assumed.
    '''
    base = pathlib.Path(base)
    if not base.exists():
        print(f"!! {base} does not exist.")
        return
    print(f"Mounted under {base}:")
    for p in sorted(base.iterdir()):
        print(f"    {p}")

    counts = count_images_per_dir(base)
    if not counts:
        print(f"\n!! No images found anywhere under {base}. Attach both datasets (Add Input) "
              f"and re-run.")
        return

    totals = collections.Counter()
    for directory, n in counts.items():
        for ancestor in [directory, *directory.parents]:
            if ancestor == base or base in ancestor.parents:
                totals[ancestor] += n
            if ancestor == base:
                break
    print(f"\nRolled-up image counts per candidate root (largest first):")
    # Some accounts nest "Add Input" datasets under /kaggle/input/datasets/<owner>/<slug>
    # instead of the more common /kaggle/input/<slug>, hence a depth deeper than 2.
    candidates = {d: n for d, n in totals.items()
                  if len(d.relative_to(base).parts) <= max_depth and n > 0}
    for directory, n in sorted(candidates.items(), key=lambda kv: (-kv[1], str(kv[0])))[:max_rows]:
        print(f"    {n:>8} images   {directory}")


def preview(root):
    '''Prints the image counts of one chosen root.'''
    root = pathlib.Path(root)
    if not root.exists():
        print(f"  !! {root} does not exist")
        return 0
    counts = count_images_per_dir(root)
    total = sum(counts.values())
    for directory, n in sorted(counts.items(), key=lambda kv: (-kv[1], str(kv[0])))[:8]:
        rel = directory.relative_to(root) if directory != root else pathlib.Path(".")
        print(f"    {str(rel):<52} {n:>8} images")
    if len(counts) > 8:
        print(f"    ... and {len(counts) - 8} more directories")
    print(f"    {'TOTAL':<52} {total:>8} images")
    return total


describe_mounts()
print(f"\nREAL_DIR = {REAL_DIR}")
n_real = preview(REAL_DIR)
print(f"\nFAKE_DIR = {FAKE_DIR}")
n_fake = preview(FAKE_DIR)

if n_real == 0 or n_fake == 0:
    # Deliberately a RuntimeError rather than SystemExit: IPython swallows SystemExit, so a
    # "Run All" would carry on into the following sections with unusable paths.
    raise RuntimeError(
        "REAL_DIR and/or FAKE_DIR hold no images. Copy the correct paths from the "
        "'Rolled-up image counts' list above into section 0, RE-RUN SECTION 0 so the new "
        "values take effect, then re-run this cell."
    )

real_path, fake_path = pathlib.Path(REAL_DIR).resolve(), pathlib.Path(FAKE_DIR).resolve()
assert real_path != fake_path, "REAL_DIR and FAKE_DIR point at the same directory."
# Nesting one root inside the other would silently pull one class's images into the other.
assert fake_path not in real_path.parents and real_path not in fake_path.parents, (
    f"One of REAL_DIR/FAKE_DIR contains the other ({real_path} / {fake_path}). Point them at "
    f"two sibling directories."
)

print(f"\nOK - {n_real} authentic and {n_fake} generated images.")

## 3 · Download & validate the phase-one CvT MFM checkpoint

In [ ]:
import gc
import hashlib

from spai.config import get_config
from spai.models.cvt import build_cvt
from spai.utils import extract_mfm_encoder_state_dict

WEIGHT_PATH.parent.mkdir(parents=True, exist_ok=True)
if not WEIGHT_PATH.exists():
    run_module("gdown", "--fuzzy", WEIGHT_URL, "-O", WEIGHT_PATH)

digest = hashlib.sha256()
with WEIGHT_PATH.open("rb") as checkpoint_file:
    for chunk in iter(lambda: checkpoint_file.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f"Unexpected checkpoint SHA-256: {actual_sha256}"

checkpoint = torch.load(WEIGHT_PATH, map_location="cpu", weights_only=False)
assert checkpoint["config"].MODEL.TYPE == "cvt"
assert checkpoint["config"].MODEL.CVT.NAME == "cvt_13"
encoder_state = extract_mfm_encoder_state_dict(checkpoint["model"])
assert len(encoder_state) == 455, f"Expected 455 CvT encoder entries, got {len(encoder_state)}"

cvt_config = get_config({"cfg": "configs/spai_cvt.yaml"})
backbone = build_cvt(cvt_config)
backbone.load_state_dict(encoder_state, strict=True)
print(f"VALID CVT MFM CHECKPOINT | epoch={checkpoint.get('epoch')} | "
      f"entries={len(encoder_state)} | sha256={actual_sha256}")
del checkpoint, encoder_state, backbone
gc.collect()

## 4 · Data audit — is there a shortcut in the data?

**This is one of the most important methodological checks in this notebook.** SPAI is meant to
detect the *spectral* inconsistencies a generative model introduces. If the authentic and
generated images also differ in resolution, file format or colour mode, a classifier can reach
a high AUC by detecting **that** instead, and the number would say nothing about
generated-image detection.

The cell below reports the acquisition properties of both source datasets. Sections 5 and 8
then remove whatever it finds using `TARGET_SIZE` / `RESIZE_MODE` / `RECODE` / `TO_GRAY` from
section 0.

In [ ]:
run_module(
    "spai.tools.prepare_medical_dataset",
    "--real-dir", REAL_DIR, "--fake-dir", FAKE_DIR,
    "-o", "/tmp/unused",
    "--audit-sample", 800,
    "--audit-only",
)

## 5 · Build the pilot dataset

Arranges a small, balanced, parity-matched subset into the layout the training code expects:

```
datasets/pilot/
├── train/{0_real, 1_fake}/
├── val/{0_real, 1_fake}/
└── test/{0_real, 1_fake}/
```

then writes the `image,class,split` CSVs with the repository's own `create_dir_csv` tool.

In [ ]:
pilot_counts = build_dataset(PILOT_DS, PILOT_MAX_PER_CLASS, "pilot")
PILOT_TRAIN_IMAGES = len(pilot_counts["train_val"].query("split == 'train'"))

## 6 · Smoke test

Builds each architecture, loads the CvT weights, and pushes a tensor of every shape the
training loop produces through it - for **both** radius configurations. Takes well under a
minute and catches integration problems before any GPU time is spent on the pilot run.

In [ ]:
import logging

from torch import nn

from spai.config import get_custom_config
from spai.models import build_cls_model
from spai.utils import load_pretrained

logging.basicConfig(level=logging.INFO, format="%(message)s")
smoke_log = logging.getLogger("smoke")


def smoke_test(radius_config):
    cfg = get_custom_config(radius_config["cfg"])
    cfg.defrost(); cfg.PRETRAINED = str(WEIGHT_PATH); cfg.freeze()

    model = build_cls_model(cfg)
    load_pretrained(cfg, model.get_vision_transformer(), smoke_log)

    n_all = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"parameters: {n_all / 1e6:.2f}M total | {n_trainable / 1e6:.2f}M trainable | "
          f"{(n_all - n_trainable) / 1e6:.2f}M frozen backbone")

    # The frozen backbone must stay in eval mode even while the model trains: CvT normalises
    # its convolutional projections with batch normalisation, which would otherwise update its
    # running statistics from batch data and mutate the frozen spectral model every step.
    model.train()
    frozen_bns = [m for m in model.get_vision_transformer().modules()
                  if isinstance(m, nn.BatchNorm2d)]
    assert not any(m.training for m in frozen_bns), "The frozen CvT backbone was left in training mode."
    print(f"batch-norm: {len(frozen_bns)} layers, all held in eval mode under model.train() -> OK")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    # Training shape: DATA.AUGMENTED_VIEWS views concatenated horizontally.
    views = cfg.DATA.AUGMENTED_VIEWS
    out = model(torch.rand(2, 3, 224, 224 * views, device=device))
    loss = nn.BCEWithLogitsLoss()(out.squeeze(1), torch.tensor([0., 1.], device=device))
    loss.backward()
    grad_norm = sum(p.grad.norm() ** 2 for p in model.parameters() if p.grad is not None) ** 0.5
    print(f"train pass : logits {tuple(out.shape)} | loss {loss.item():.4f} | grad-norm {grad_norm:.2f}")
    if cfg.MODEL.FRE.LEARNABLE_MASKING_RADIUS:
        print(f"masking radius after one step: {model.get_masking_radius():.4f}")

    # Inference shape: a list of images of arbitrary and differing resolutions.
    model.eval()
    with torch.no_grad():
        out = model(
            [torch.rand(1, 3, 512, 512, device=device), torch.rand(1, 3, 448, 672, device=device)],
            FEATURE_EXTRACTION_BATCH,
        )
    scores = [round(s, 4) for s in torch.sigmoid(out).squeeze(1).tolist()]
    print(f"infer pass : logits {tuple(out.shape)} | scores {scores}")

    del model
    torch.cuda.empty_cache()
    print("SMOKE TEST PASSED")


run_for_each_config(RADIUS_CONFIGS, smoke_test)

## 7 · Pilot run

A couple of epochs on a few hundred images, for **both** radius configurations. **This is the
gate** - if it completes and the loss moves, the full run will work. If it raises, stop here and
fix it before spending GPU hours on section 8.

In [ ]:
def pilot_train(radius_config):
    output = PILOT_OUT / radius_config["key"]
    shutil.rmtree(output, ignore_errors=True)  # the pilot always starts fresh
    run_module("spai", *train_cmd(
        radius_config["cfg"], PILOT_DS / "train_val.csv", PILOT_DS,
        output, PILOT_TAG,
        PILOT_EPOCHS, PILOT_WARMUP, radius_config["pilot_batch"], radius_config["pilot_accum"],
        PILOT_LR, FULL_AMP, save_all=True,
    ))
    checkpoint_dir = run_dir(output, radius_config["model_name"], PILOT_TAG)
    assert list(checkpoint_dir.glob("ckpt_epoch_*.pth")), f"No checkpoint written to {checkpoint_dir}"
    radius_config["pilot_run_dir"] = checkpoint_dir


run_for_each_config(RADIUS_CONFIGS, pilot_train)

### 7.1 Pilot learning curves

In [ ]:
def summarize_pilot():
    fig, (ax_loss, ax_auc) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
    for radius_config, color in zip(RADIUS_CONFIGS, ("#2a78d6", "#eb6834")):
        history = parse_training_log(radius_config["pilot_run_dir"])
        radius_config["pilot_history"] = history
        ax_loss.plot(history.epoch, history.val_loss, marker="o", color=color,
                     label=radius_config["label"])
        ax_auc.plot(history.epoch, history.val_auc, marker="o", color=color,
                    label=radius_config["label"])
    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("validation loss")
    ax_loss.set_title("Pilot validation loss", loc="left"); ax_loss.legend()
    ax_auc.set_xlabel("epoch"); ax_auc.set_ylabel("validation AUC")
    ax_auc.set_title("Pilot validation AUC", loc="left"); ax_auc.legend()
    plt.show()

    for radius_config in RADIUS_CONFIGS:
        h = radius_config["pilot_history"]
        print(f"{radius_config['label']:<18} final val loss {h.val_loss.iloc[-1]:.4f} | "
              f"val AUC {h.val_auc.iloc[-1]:.4f} | epoch time {h.epoch_time.iloc[-1]:.0f}s")

    learnable = next(c for c in RADIUS_CONFIGS if c["key"] == "learnable")
    if "masking_radius" in learnable["pilot_history"]:
        h = learnable["pilot_history"]
        print(f"\nLearnable radius: started at {h.masking_radius.iloc[0]:.4f}, "
              f"reached {h.masking_radius.iloc[-1]:.4f} after {PILOT_EPOCHS} pilot epochs.")


summarize_pilot()

> **Stop here if either pilot run failed, or produced a NaN/exploding loss.** Report the
> traceback and it can be fixed before you spend GPU hours. If both completed, the numbers do
> not need to be good yet - two epochs on a couple hundred images only proves the pipeline
> runs end to end.

## 8 · Full run

### Checkpointing & resuming

This repository's training CLI already checkpoints safely and resumes automatically - nothing
extra to enable:

- Every checkpoint is written to a temporary file and atomically moved into place
  (`ckpt_epoch_N.pth`), so an interrupted write can never be picked up as a valid checkpoint.
- Checkpoints are saved on **every epoch that improves validation loss** (so the
  highest-numbered checkpoint in a run's directory is always its best one). This bounds disk
  usage automatically for the full run; the pilot's `--save-all` (which keeps every epoch
  regardless) is only used because pilot runs are short.
- When `--resume` is **not** passed, the CLI automatically looks for the newest
  `ckpt_epoch_*.pth` already sitting in `<output>/<model_name>/<tag>` and resumes from it
  (`TRAIN.AUTO_RESUME`, on by default). This is why `FULL_TAG` in section 0 is a **fixed
  string, not a timestamp** - re-running a training cell below with no changes continues from
  the last completed epoch instead of starting over.

In practice:
- **Interrupted mid-session** (kernel restart, OOM, you stopped it): just re-run the training
  cell. `/kaggle/working` survived, so auto-resume picks it up.
- **Continuing in a brand-new Kaggle session** (the previous Commit finished or expired, so
  `/kaggle/working` is empty again): attach that Version's output as a new **Input** dataset,
  set `PREVIOUS_FIXED_CHECKPOINT` / `PREVIOUS_LEARNABLE_CHECKPOINT` in section 0 to its
  `ckpt_epoch_*.pth` file, and re-run the notebook from the top - the training cell passes it
  via `--resume` explicitly.

### One radius configuration per session

Run **one** of the two training cells below per Kaggle session/Commit, not both back to back -
each can run for hours, and Kaggle GPU sessions are time-boxed (the projection below estimates
how long). Skip the cell for the configuration you are not training this session.

### 8.1 Build the full dataset

In [ ]:
# Rough disk estimate: a 512x512 grayscale PNG is roughly 0.15-0.25 MB.
if FULL_MAX_PER_CLASS is not None:
    estimated_gb = 2 * FULL_MAX_PER_CLASS * 0.22 / 1024
    free_gb = shutil.disk_usage(WORK).free / 1024 ** 3
    print(f"estimated dataset size: {estimated_gb:.1f} GiB | free: {free_gb:.1f} GiB")
    assert estimated_gb < free_gb * 0.7, "Not enough free space - lower FULL_MAX_PER_CLASS in section 0."

shutil.rmtree(PILOT_DS, ignore_errors=True)  # no longer needed, frees disk for the full dataset
full_counts = build_dataset(FULL_DS, FULL_MAX_PER_CLASS, "full")
FULL_TRAIN_IMAGES = len(full_counts["train_val"].query("split == 'train'"))

### 8.2 Projected duration

Catches a schedule that would not fit inside one Kaggle GPU session, before committing to it.

In [ ]:
print(f"pilot: {PILOT_TRAIN_IMAGES} train images")
print(f"full : {FULL_TRAIN_IMAGES} train images\n")

for radius_config in RADIUS_CONFIGS:
    pilot_epoch_time = radius_config["pilot_history"].epoch_time.iloc[-1]
    # Epoch time is dominated by the number of optimizer steps, and pilot/full batch sizes can
    # differ (the learnable config), so scale by steps rather than raw image counts.
    pilot_steps = PILOT_TRAIN_IMAGES / radius_config["pilot_batch"]
    full_steps = FULL_TRAIN_IMAGES / radius_config["full_batch"]
    per_epoch = pilot_epoch_time * full_steps / pilot_steps
    total_hours = per_epoch * FULL_EPOCHS / 3600
    print(f"{radius_config['label']:<18} pilot {pilot_epoch_time:.0f}s/epoch @ batch "
          f"{radius_config['pilot_batch']} -> full ~{per_epoch / 60:.1f} min/epoch @ batch "
          f"{radius_config['full_batch']} -> ~{total_hours:.1f}h for {FULL_EPOCHS} epochs")
    if total_hours > 8:
        print("  [WARNING] That may not fit in a single Kaggle GPU session. Lower FULL_EPOCHS "
              "/ FULL_MAX_PER_CLASS in section 0, or rely on checkpoint auto-resume across "
              "multiple sessions/Commits (see the note above) - an overrun is recoverable, "
              "not fatal.")

### 8.3 Train — fixed radius

In [ ]:
fixed_config = next(c for c in RADIUS_CONFIGS if c["key"] == "fixed")
fixed_output = FULL_OUT / "fixed"

run_module("spai", *train_cmd(
    fixed_config["cfg"], FULL_DS / "train_val.csv", FULL_DS,
    fixed_output, FULL_TAG,
    FULL_EPOCHS, FULL_WARMUP, fixed_config["full_batch"], fixed_config["full_accum"],
    FULL_LR, FULL_AMP, resume=fixed_config["previous_checkpoint"],
))

print("\nCheckpoints:", sorted(p.name for p in full_run_dir(fixed_config).glob("ckpt_epoch_*.pth")))
plot_history(full_run_dir(fixed_config), "Fixed radius — full run")

### 8.4 Train — learnable radius

In [ ]:
learnable_config = next(c for c in RADIUS_CONFIGS if c["key"] == "learnable")
learnable_output = FULL_OUT / "learnable"

run_module("spai", *train_cmd(
    learnable_config["cfg"], FULL_DS / "train_val.csv", FULL_DS,
    learnable_output, FULL_TAG,
    FULL_EPOCHS, FULL_WARMUP, learnable_config["full_batch"], learnable_config["full_accum"],
    FULL_LR, FULL_AMP, resume=learnable_config["previous_checkpoint"],
))

print("\nCheckpoints:", sorted(p.name for p in full_run_dir(learnable_config).glob("ckpt_epoch_*.pth")))
history = plot_history(full_run_dir(learnable_config), "Learnable radius — full run")
if "masking_radius" in history:
    print(f"\nLearned radius: started at {history.masking_radius.iloc[0]:.4f}, "
          f"reached {history.masking_radius.iloc[-1]:.4f}.")

## 9 · Evaluation on the held-out test split

Evaluates whichever configuration(s) have a checkpoint on disk under `FULL_OUT` - regardless
of whether section 8's training cell ran to completion, was stopped early, or ran in an
earlier session; only training at least one epoch (so one `ckpt_epoch_*.pth` exists) matters -
on their held-out test split, seen during neither training nor model selection.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score, roc_curve


def evaluate_on_test(radius_config):
    checkpoint = latest_checkpoint(radius_config["full_run_dir"])
    epoch = int(re.search(r"ckpt_epoch_(\d+)", checkpoint.name).group(1))
    tag = f"full_{radius_config['key']}"
    print(f"evaluating {checkpoint.name} (epoch {epoch} - the best checkpoint, since "
          f"checkpoints are only saved when validation loss improves)")

    test_output = WORK / "output" / "test"
    run_module(
        "spai", "test",
        "--cfg", radius_config["cfg"],
        "--batch-size", VAL_BATCH,
        "--model", checkpoint,
        "--output", test_output,
        "--tag", tag,
        "--test-csv", FULL_DS / "test.csv",
        "--test-csv-root-dir", FULL_DS,
        "--update-csv",
        "--opt", "DATA.NUM_WORKERS", str(DATA_WORKERS),
        "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH),
        "--opt", "DATA.TEST_PREFETCH_FACTOR", "1",
    )

    scored_csv = run_dir(test_output, radius_config["model_name"], tag) / "test.csv"
    score_column = f"{tag}_epoch_{epoch}"
    scored = pd.read_csv(scored_csv)
    scored = scored[pd.to_numeric(scored[score_column], errors="coerce").notna()]
    radius_config["test_scores"] = scored
    radius_config["score_column"] = score_column
    radius_config["scored_csv"] = scored_csv


def plot_test_results(radius_config):
    scored = radius_config["test_scores"]
    y = scored["class"].astype(int).to_numpy()
    s = scored[radius_config["score_column"]].astype(float).to_numpy()
    auc = roc_auc_score(y, s)
    ap = average_precision_score(y, s)
    acc = accuracy_score(y, (s >= 0.5).astype(int))

    fig, (ax_hist, ax_roc) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
    bins = np.linspace(0, 1, 41)
    ax_hist.hist(s[y == 0], bins=bins, alpha=0.75, label="authentic", color="#2a78d6")
    ax_hist.hist(s[y == 1], bins=bins, alpha=0.75, label="generated", color="#eb6834")
    ax_hist.axvline(0.5, color="#52514e", lw=1, ls="--")
    ax_hist.set_xlabel("predicted score"); ax_hist.set_ylabel("images")
    ax_hist.set_title("Score distribution", loc="left"); ax_hist.legend()

    fpr, tpr, _ = roc_curve(y, s)
    ax_roc.plot([0, 1], [0, 1], color="#e2e1dd", lw=1.5, ls="--")
    ax_roc.plot(fpr, tpr, color="#2a78d6", lw=2)
    ax_roc.annotate(f"AUC {auc:.3f}", (0.55, 0.18), fontsize=11)
    ax_roc.set_xlabel("false positive rate"); ax_roc.set_ylabel("true positive rate")
    ax_roc.set_title("ROC", loc="left"); ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.02)

    fig.suptitle(f"{radius_config['label']} — held-out test split", x=0.01, ha="left")
    plt.show()

    print(f"images   : {len(y)}  ({int((y == 0).sum())} authentic / {int((y == 1).sum())} generated)")
    print(f"AUC      : {auc:.4f}")
    print(f"AP       : {ap:.4f}")
    print(f"Accuracy : {acc:.4f}  (threshold 0.5)")


for radius_config in RADIUS_CONFIGS:
    radius_config["full_run_dir"] = full_run_dir(radius_config)
    if not list(radius_config["full_run_dir"].glob("ckpt_epoch_*.pth")):
        print(f"Skipping {radius_config['label']}: no checkpoint under "
              f"{radius_config['full_run_dir']} (see section 8).")
        continue
    evaluate_on_test(radius_config)
    plot_test_results(radius_config)

### 9.1 Shortcut probe — could a deliberately weak model do just as well?

A near-perfect AUC only counts as evidence of **spectral artifact** detection if the two
classes are not already separable by something far cruder. This cell trains two deliberately
weak baselines on exactly the split SPAI used:

1. **16x16 thumbnail** — each image is reduced to 256 pixels, which throws away essentially all
   high-frequency content. Anything still separable here is gross layout, brightness and
   contrast, not a generative fingerprint.
2. **Intensity statistics** — mean, standard deviation and seven percentiles per image. No
   spatial structure at all, just the shape of the intensity histogram.

Both feed a plain logistic regression, and the table at the end puts them next to SPAI's own
score on the same test images. How to read it:

| Weak-probe AUC | Interpretation |
|---|---|
| >= 0.95 | The classes are separable with no spectral information whatsoever. SPAI's score cannot be read as evidence that it detects generative artifacts — the two datasets simply differ. Report this alongside the AUC. |
| 0.75 – 0.95 | A substantial shortcut exists, though SPAI may add something on top of it. Report both numbers. |
| < 0.75 | Crude statistics do not explain the result. SPAI's margin over the probe is the meaningful quantity. |

Takes a couple of minutes — it reads every training and test image once.

In [ ]:
import numpy as np
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

THUMBNAIL_SIZE = 16

assert (FULL_DS / "train_val.csv").exists() and (FULL_DS / "test.csv").exists(), (
    f"The full dataset is missing from {FULL_DS}. Re-run section 8.1 to rebuild it."
)


def load_probe_features(csv_path, split):
    '''Reads every image of one split once, returning both weak feature sets and the labels.'''
    df = pd.read_csv(csv_path)
    df = df[df["split"] == split].reset_index(drop=True)
    thumbnails = np.zeros((len(df), THUMBNAIL_SIZE * THUMBNAIL_SIZE), dtype=np.float32)
    statistics = np.zeros((len(df), 9), dtype=np.float32)

    for i, relative_path in enumerate(df["image"]):
        with Image.open(FULL_DS / relative_path) as image:
            gray = image.convert("L")
            thumbnails[i] = np.asarray(
                gray.resize((THUMBNAIL_SIZE, THUMBNAIL_SIZE), Image.BICUBIC), dtype=np.float32
            ).ravel() / 255.0
            # Every 4th pixel is ample for an intensity histogram and keeps this loop fast.
            pixels = np.asarray(gray, dtype=np.float32)[::4, ::4].ravel() / 255.0
        statistics[i] = [pixels.mean(), pixels.std(),
                         *np.percentile(pixels, [1, 5, 25, 50, 75, 95, 99])]
        if (i + 1) % 2000 == 0:
            print(f"    {i + 1}/{len(df)} {split} images")

    return thumbnails, statistics, df["class"].to_numpy(dtype=int)


def run_probe(name, train_features, train_labels, test_features, test_labels):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    model.fit(train_features, train_labels)
    scores = model.predict_proba(test_features)[:, 1]
    return {
        "probe": name,
        "AUC": roc_auc_score(test_labels, scores),
        "AP": average_precision_score(test_labels, scores),
        "accuracy": accuracy_score(test_labels, (scores >= 0.5).astype(int)),
    }


def spai_result(radius_config):
    '''Re-reads SPAI's own per-image test scores from disk, so no in-memory state is needed.'''
    tag = f"full_{radius_config['key']}"
    scored_csv = run_dir(WORK / "output" / "test", radius_config["model_name"], tag) / "test.csv"
    if not scored_csv.exists():
        return None
    scored = pd.read_csv(scored_csv)
    columns = [c for c in scored.columns if re.fullmatch(re.escape(tag) + r"_epoch_\d+", c)]
    if not columns:
        return None
    column = max(columns, key=lambda c: int(re.search(r"_epoch_(\d+)$", c).group(1)))
    scored = scored[pd.to_numeric(scored[column], errors="coerce").notna()]
    labels = scored["class"].astype(int).to_numpy()
    scores = scored[column].astype(float).to_numpy()
    epoch = re.search(r"_epoch_(\d+)$", column).group(1)
    return {
        "probe": f"SPAI - {radius_config['label']} (epoch {epoch})",
        "AUC": roc_auc_score(labels, scores),
        "AP": average_precision_score(labels, scores),
        "accuracy": accuracy_score(labels, (scores >= 0.5).astype(int)),
    }


print("Reading training images...")
train_thumbnails, train_statistics, train_labels = load_probe_features(
    FULL_DS / "train_val.csv", "train"
)
print("Reading test images...")
test_thumbnails, test_statistics, test_labels = load_probe_features(FULL_DS / "test.csv", "test")
print(f"\ntrain: {len(train_labels)} images | test: {len(test_labels)} images\n")

results = [
    run_probe(f"{THUMBNAIL_SIZE}x{THUMBNAIL_SIZE} thumbnail + logistic regression",
              train_thumbnails, train_labels, test_thumbnails, test_labels),
    run_probe("intensity statistics + logistic regression",
              train_statistics, train_labels, test_statistics, test_labels),
]
results += [r for r in (spai_result(c) for c in RADIUS_CONFIGS) if r is not None]
print(pd.DataFrame(results).set_index("probe").round(4).to_string())

probe_auc = max(r["AUC"] for r in results if "logistic regression" in r["probe"])
print()
if probe_auc >= 0.95:
    print(f">>> The weak probes reach {probe_auc:.4f} AUC with no access to spectral detail at\n"
          f"    all. The two classes are separable on crude image statistics alone, so SPAI's\n"
          f"    score cannot be read as evidence that it detects generative artifacts. This\n"
          f"    belongs in the write-up next to the headline AUC.")
elif probe_auc >= 0.75:
    print(f">>> The weak probes reach {probe_auc:.4f} AUC. A substantial shortcut exists, though\n"
          f"    SPAI may still be adding something beyond it. Report both numbers together.")
else:
    print(f">>> The weak probes only reach {probe_auc:.4f} AUC, so crude statistics do not\n"
          f"    explain SPAI's result. Its margin over the probe is the meaningful quantity.")

## 10 · Save artifacts

In [ ]:
artifacts_dir = WORK / "artifacts"
artifacts_dir.mkdir(exist_ok=True)

for radius_config in RADIUS_CONFIGS:
    radius_config["full_run_dir"] = full_run_dir(radius_config)
    if not list(radius_config["full_run_dir"].glob("ckpt_epoch_*.pth")):
        continue
    checkpoint = latest_checkpoint(radius_config["full_run_dir"])
    shutil.copy(checkpoint, artifacts_dir / f"spai_cvt_{radius_config['key']}_best.pth")
    for name in ("log_rank0.txt", "config.json"):
        src = radius_config["full_run_dir"] / name
        if src.exists():
            shutil.copy(src, artifacts_dir / f"{radius_config['key']}_{name}")
    if "scored_csv" in radius_config:
        shutil.copy(radius_config["scored_csv"],
                    artifacts_dir / f"{radius_config['key']}_test_scores.csv")

archive_path = shutil.make_archive(str(WORK / "spai_cvt_results"), "gztar", root_dir=artifacts_dir)
print("RESULT ARCHIVE:", archive_path)
for p in sorted(artifacts_dir.iterdir()):
    print(f"  {p.name:<32} {p.stat().st_size / 1024 ** 2:8.2f} MB")

---
## Reading the result

- **Domain shift.** The spectral model `G` was pre-trained on natural images (an ImageNet
  subset) but is used here to model the spectral distribution of chest X-rays. The paper's
  premise is that `G` captures the spectral distribution of *real* images; a natural-image `G`
  may model X-ray spectra only approximately. If the AUC disappoints, the highest-value
  follow-up is repeating the phase-one MFM pre-training on real X-rays.
- **Grayscale.** X-rays carry no chrominance, so the colour-related spectral cues available in
  the paper's setting are absent here. Expect a lower absolute AUC than the paper's reported
  91.0 on natural images.
- **A single generator.** With one synthetic source, this measures in-domain detection only. It
  does **not** test SPAI's central claim of generalizing to unseen generators - that would need
  a second generator held out entirely from training.
- **Parity.** Whatever section 4 reported, state it in the write-up alongside the
  transformations applied in sections 5/8. The first question a reader asks about any
  AI-generated-image detector is whether the two classes differ in some trivial way.
- **The ablation.** Comparing sections 8.3 and 8.4 on the identical split is the fixed-vs-
  learnable radius comparison this fork exists to run - see
  [docs/physics_informed_frequency_masking.md](docs/physics_informed_frequency_masking.md) for
  the motivation and [docs/learnable_radius.md](docs/learnable_radius.md) for the implementation.